# 03 — 実験結果の比較・可視化

## 1. Overview

- 目的：推薦性能・ユーザー間の差・効率を比較し、研究上の結論を整理する。
- 入力：保存済みの評価結果。Benchmarkの実行・評価指標の計算は `src/` で行う。
- 既定：合成データ `toy-music-v1` のPopularity保存結果（5要求、評価対象4要求）。
- Music4AllとTalkPlayDataは独立して分析する。`MUSIC_RECSYS_REPORT_PATH` で実データの保存結果を指定できる。

| Notebook | 役割 |
| --- | --- |
| 01 | 保存済みデータの検証・EDA |
| 02 | 推薦を動かし、挙動を理解する |
| 03 | 保存済みの実験結果から性能を比較する |

## 2. Setup

| 設定 | 用途 |
| --- | --- |
| `MUSIC_RECSYS_DATASET_ID` / `MUSIC_RECSYS_TOP_K` | 評価対象・Kを上書き |
| `MUSIC_RECSYS_REPORT_PATH` | Popularityの保存済み `results.json`。指定時の既定はMusic4All・K=10 |
| `REPORT_PATHS` | 複数runを比べる場合は表示名とPathを追加 |
| `MLFLOW_RUNS` | 任意：表示名とMLflow run ID |
| `DETAILS_ROOT` | 任意：MLflowのmanifestが参照するローカル保存先の親ディレクトリ |
| `BASELINE` | 差分の基準となるrun |
| `BOOTSTRAP` | 信頼水準・再標本化回数・乱数seed |
| `HISTORY_EDGES` / `USER_ATTRIBUTE` | 履歴量の区切り・属性名 |

- 設定変更後はこの章から実行。MLflowは読込みだけを行う。
- 比較前にデータ・分割・入力・候補・K・評価器・集計ルールを照合。不一致はエラー。
- 古い結果で評価条件の識別情報が不足する場合は、CLIで再実行・保存してから指定する。
- 読込み・条件照合・統計処理は `src/music_recsys/experiment_analysis.py` を呼ぶ。

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from music_recsys.evaluation.tracking import TrackingConfig
from music_recsys.experiment_analysis import (
    BootstrapConfig,
    additional_metric_rows,
    condition_rows,
    efficiency_rows,
    failure_rows,
    group_rows,
    load_reports,
    paired_differences,
    performance_rows,
    read_mlflow_report,
    repetition_rows,
    user_rows,
)

ROOT = Path.cwd() if Path("src").exists() else Path.cwd().parent
saved_report_path = os.environ.get("MUSIC_RECSYS_REPORT_PATH")
DATASET_ID = os.environ.get(
    "MUSIC_RECSYS_DATASET_ID", "music4all-onion" if saved_report_path else "toy-music-v1"
)
TOP_K = int(os.environ.get("MUSIC_RECSYS_TOP_K", "10" if saved_report_path else "2"))
REPORT_PATHS = {
    "Popularity": Path(saved_report_path)
    if saved_report_path
    else ROOT / "examples/reports/toy-popularity.json"
}
MLFLOW_RUNS = {}
DETAILS_ROOT = None
BASELINE = "Popularity"
METRICS = ("ndcg", "recall")
DETAIL_METRIC = "recall"
BOOTSTRAP = BootstrapConfig(confidence=0.95, resamples=2000, seed=42)
HISTORY_EDGES = (0, 1, 5, 20)
USER_ATTRIBUTE = None

In [ ]:
sources = dict(REPORT_PATHS)
if MLFLOW_RUNS:
    tracking = TrackingConfig.from_settings()
    for label, run_id in MLFLOW_RUNS.items():
        if label in sources:
            raise ValueError(f"duplicate report label: {label}")
        sources[label] = read_mlflow_report(
            run_id, tracking_uri=tracking.tracking_uri, details_root=DETAILS_ROOT
        )
reports = load_reports(sources, dataset_id=DATASET_ID, top_k=TOP_K)
conditions_df = pd.DataFrame(condition_rows(reports))
display(
    conditions_df[["run", "model_name", "seed", "dataset_id", "top_k", "started_at", "details"]]
)
display(
    conditions_df[["run", "dataset_sha256", "source_sha256", "config_sha256", "model_parameters"]]
)
common_conditions = next(iter(reports.values()))["metadata"]["evaluation_conditions"]
display(pd.DataFrame(common_conditions.items(), columns=["condition", "value"]))
missing_details = [label for label, raw in reports.items() if "runs" not in raw]
if missing_details:
    print("詳細なし（信頼区間・ユーザー別・失敗内訳は未分析）:", ", ".join(missing_details))

## 3. Performance

- 平均は評価対象の要求を等重みで集計。正解なしは除外し、失敗は正解があれば0として含める。
- 信頼区間：ユーザー単位のpercentile bootstrap。複数要求・会話ターンはユーザーごとまとめて再標本化。
- ユーザー2人未満・詳細結果なしでは信頼区間を表示しない。
- 合成データは少数ユーザーの動作確認。信頼区間を実データでの性能保証と解釈しない。

In [ ]:
performance_df = pd.DataFrame(
    [
        row
        for metric in METRICS
        for row in performance_rows(reports, metric=metric, bootstrap=BOOTSTRAP)
    ]
)
display(performance_df)

fig, axes = plt.subplots(1, len(METRICS), figsize=(6 * len(METRICS), 3.5), squeeze=False)
for ax, metric in zip(axes.flat, METRICS):
    frame = performance_df[performance_df["metric"] == metric].reset_index(drop=True)
    valid = frame[frame["mean"].notna()]
    ax.scatter(valid.index, valid["mean"], color="#267d85", label="Request mean")
    intervals = frame.dropna(subset=["ci_low", "ci_high"])
    ax.vlines(intervals.index, intervals["ci_low"], intervals["ci_high"], color="#267d85")
    ax.set_xticks(frame.index, frame["run"], rotation=20)
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f"{metric.upper()}@{TOP_K} / {BOOTSTRAP.confidence:.0%} user CI")
    if valid.empty:
        ax.text(0.5, 0.5, "Undefined", ha="center", transform=ax.transAxes)
fig.tight_layout()
plt.show()

### 基準モデルとの差・反復実行のばらつき

- 対応差分：同じ要求IDで対応付け、`比較run − BASELINE` を計算。
- 差分の信頼区間もユーザー単位。複数runを指定したときに表示する。
- 反復表はモデル名・設定・実装コードが一致するrunを集計。異なるseedと同じseedの再実行を件数で区別。
- `run_std` は実行間の標準偏差。ユーザー間のばらつき・信頼区間とは別に扱う。

In [ ]:
difference_df = pd.DataFrame(
    paired_differences(reports, baseline=BASELINE, metric=DETAIL_METRIC, bootstrap=BOOTSTRAP)
)
if difference_df.empty:
    print("比較runは1件。別の保存結果をSetupに追加すると対応差分を表示します。")
else:
    display(difference_df)
repetition_df = pd.DataFrame(repetition_rows(reports, metric=DETAIL_METRIC))
display(repetition_df)

## 4. Detailed Analysis

### ユーザー別・履歴量別・属性別

- ユーザー別の平均・分布と、履歴量グループの性能を確認する。
- 履歴量は推薦時点の入力から取得。区間は左端を含み、右端を含まない。
- 属性は保存済み `user_attributes` のみ使用。未取得は `(missing)` に分類。
- グループごとに要求数・評価対象数・ユーザー数を表示。少数群の解釈に注意する。

In [ ]:
users_df = pd.DataFrame(user_rows(reports, metric=DETAIL_METRIC, baseline=BASELINE))
history_df = pd.DataFrame(
    group_rows(reports, metric=DETAIL_METRIC, history_edges=HISTORY_EDGES, bootstrap=BOOTSTRAP)
)
if users_df.empty:
    print("詳細結果なし：ユーザー別・履歴量別の分析は未実施。")
else:
    display(users_df.pivot(index="user", columns="run", values="mean"))
    display(users_df.pivot(index="user", columns="run", values="delta_from_baseline"))
    display(history_df)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    groups = [
        (label, frame["mean"].dropna().tolist())
        for label, frame in users_df.groupby("run", sort=False)
    ]
    groups = [(label, values) for label, values in groups if values]
    if groups:
        ax.boxplot([values for _, values in groups], tick_labels=[label for label, _ in groups])
    else:
        ax.text(0.5, 0.5, "No evaluated users", ha="center", transform=ax.transAxes)
    ax.set_ylabel(f"User mean {DETAIL_METRIC}@{TOP_K}")
    ax.set_ylim(-0.05, 1.05)
    fig.tight_layout()
    plt.show()

if USER_ATTRIBUTE is None:
    print("ユーザー属性は未選択。既定の合成結果には属性データがありません。")
else:
    display(
        pd.DataFrame(
            group_rows(reports, metric=DETAIL_METRIC, attribute=USER_ATTRIBUTE, bootstrap=BOOTSTRAP)
        )
    )

### Diversity・Coverage・Novelty

| 指標 | 比較時に固定する定義 |
| --- | --- |
| Diversity | 類似度・特徴量・Embeddingの版、集計方法 |
| Coverage | 対象カタログ・評価対象集合・K |
| Novelty | 人気度の参照期間・算出方法 |

- `additional_metrics` に保存された値と定義を表示。定義不一致は比較不可。
- 現在は未計算。Notebook内で指標を作らず、未測定を0に置き換えない。

In [ ]:
additional_df = pd.DataFrame(additional_metric_rows(reports))
display(additional_df)
measured = additional_df.dropna(subset=["value"])
if not measured.empty:
    for metric, frame in measured.groupby("metric", sort=False):
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.bar(frame["run"], frame["value"], color="#627d98")
        ax.set_title(metric)
        fig.tight_layout()
        plt.show()

### 失敗分析

| 分類 | 内容 |
| --- | --- |
| `error` / `error:種類` | 推薦処理の失敗・例外の種類 |
| `short_output` | 有効な推薦曲がK未満 |
| `invalid_items` / `duplicate_items` | 無効な曲・重複曲を含む要求 |
| `no_relevant_candidates` | 候補内に正解なし。評価対象外 |
| `zero_recall` | 評価対象だが正解曲を推薦できなかった要求 |

- 分類は重複するため合計しない。出力不足は候補不足の場合も含む。

In [ ]:
failure_summary_df = pd.DataFrame(
    [
        {
            "run": label,
            **{
                key: raw["summary"][key]
                for key in ("n_requests", "n_failed", "n_short", "n_excluded")
            },
        }
        for label, raw in reports.items()
    ]
)
display(failure_summary_df)
failures_df = pd.DataFrame(failure_rows(reports))
if failures_df.empty:
    print("詳細結果なし：失敗の内訳は未分析。集計済み件数は上の表で確認できます。")
else:
    display(failures_df)
    ax = failures_df.pivot(index="category", columns="run", values="count").plot.barh(
        figsize=(9, 4)
    )
    ax.set_xlabel("Requests (overlapping categories)")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

## 5. Efficiency

- 学習時間、推論時間の平均・中央値・p95、推論コストを比較。
- コストは全要求分の金額・通貨・算定基準が揃う場合のみ合計する。欠測を無料と扱わない。
- 実行環境・並列数・キャッシュ等の条件を確認する。未記録の条件は推測しない。
- 散布図は性能との関係を探索するもの。合成データの短い計測から速度の優劣を結論付けない。

In [ ]:
efficiency_df = pd.DataFrame(efficiency_rows(reports))
display(efficiency_df)
tradeoff_df = performance_df[performance_df["metric"] == DETAIL_METRIC].merge(
    efficiency_df, on="run"
)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
latency = tradeoff_df.dropna(subset=["latency_mean", "mean"])
axes[0].scatter(latency["latency_mean"], latency["mean"], color="#267d85")
for _, row in latency.iterrows():
    axes[0].annotate(row["run"], (row["latency_mean"], row["mean"]))
axes[0].set_xlabel("Mean inference time (s / request)")
axes[0].set_ylabel(f"{DETAIL_METRIC}@{TOP_K}")
axes[0].set_ylim(-0.05, 1.05)

costs = tradeoff_df.dropna(subset=["cost_per_request", "mean"])
units = costs[["cost_currency", "cost_basis"]].drop_duplicates()
if costs.empty:
    axes[1].text(0.5, 0.5, "Cost not measured", ha="center", transform=axes[1].transAxes)
elif len(units) != 1:
    axes[1].text(0.5, 0.5, "Different cost units / bases", ha="center", transform=axes[1].transAxes)
else:
    axes[1].scatter(costs["cost_per_request"], costs["mean"], color="#b77742")
    for _, row in costs.iterrows():
        axes[1].annotate(row["run"], (row["cost_per_request"], row["mean"]))
    axes[1].set_xlabel(f"Cost ({units.iloc[0]['cost_currency']} / request)")
axes[1].set_ylabel(f"{DETAIL_METRIC}@{TOP_K}")
axes[1].set_ylim(-0.05, 1.05)
fig.tight_layout()
plt.show()

## 6. Summary

- 既定の結果はPopularity 1モデル・合成データ。モデル間の優劣や実データでの有効性は未検証。
- 既定の保存結果で未測定：ユーザー属性、Diversity・Coverage・Novelty、推論コスト。
- LLM・Agentも共通形式の保存結果を追加して比較する。追加指標の定義は評価側で管理する。

| 整理する観点 | 結果・根拠・限界 |
| --- | --- |
| 仮説への回答 | — |
| 推薦性能とばらつき | — |
| 改善・悪化したユーザー層 | — |
| 失敗傾向・多様性・発見性 | — |
| 性能と時間・コストのトレードオフ | — |
| 研究上の結論・次の検証 | — |

- 共有時は出力を消去し、データ・run・分析設定を残す。